# Pandas Interview Recap — Mini Notebook (Go-To)
Fast recap before a Pandas coding interview.

**Focus**
- Common patterns (filter/assign, groupby, windows, reshape)
- Time + strings
- Pivot MultiIndex column flattening
- Lightweight plotting

## 0) Imports + tiny helpers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)



## 1) `loc` vs `iloc` (and safe assignment)
- **`loc`**: label-based + boolean masks; label slices are **inclusive**
- **`iloc`**: position-based; end is **exclusive**
- Avoid chained assignment: use `.loc[...]` + `.copy()`

In [ ]:
# loc: label-based / boolean filter
df.loc[df["country"] == "IN", ["user_id","revenue"]]


# safe assignment
df2 = df.loc[df["x"] > 0].copy()
df2.loc[:, "ratio"] = df2["a"] / df2["b"]


# iloc: position-based
df.iloc[:10, [0, 3, 5]]

## 2) Quick inspect / sanity

In [ ]:
df.shape
df.head()
df.sample(5, random_state=0)
df.info()
df.describe(include="all")
df.isna().sum().sort_values(ascending=False).head(20)
df.nunique().sort_values(ascending=False).head(20)

## 4) Sorting + dedup patterns

In [ ]:
df = df.sort_values(["user_id","ts"], ascending = [False, True] )

latest per key
latest = (df.sort_values("ts")
            .drop_duplicates("user_id", keep="last"))

## 3)  Rename / drop / dtype fixes

In [ ]:
Rename
df = df.rename(columns={"old": "new"})
df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True))

# Drop
df = df.drop(columns=["col1","col2"])
#removes all rows from the DataFrame df that have a missing value
df = df.dropna(subset=["key_col"])
df = df.drop_duplicates(subset=["user_id"], keep="last")

# Dtypes
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df["user_id"] = df["user_id"].astype("int64")
df["country"] = df["country"].astype("category")

## 5) Groupby: `agg` vs `transform`
- `agg` reduces rows
- `transform` keeps same length (features)

In [ ]:
# as_index=False parameter in a pandas groupby() operation is used to prevent the grouped-by columns from becoming the 
#index of the resulting DataFrame after an aggregation.

out = (df.groupby("user_id")
         .agg(events=("event_id","count"),
              spend=("revenue","sum"),
              last_ts=("ts","max"))
         .reset_index())

df["user_mean_revenue"] = df.groupby("user_id")["revenue"].transform("mean")
# example of transform
# data = {'User': ['A', 'B', 'A', 'B', 'A', 'B'], 'PurchaseAmount': [10, 15, 20, 25, 30, 35]}
# df = pd.DataFrame(data)
# # Calculate the mean purchase amount for each user and assign it back to a new column
# df['MeanPurchase'] = df.groupby('User')['PurchaseAmount'].transform('mean')


# top-N per group
top3 = (df.sort_values(["user_id","revenue"], ascending=[True, False])
          .groupby("user_id").head(3))

#Top-N unique per group
result = (df.sort_values(['Group', 'Value'], ascending=[True, False])
            .drop_duplicates(subset=['Group', 'Value'])
            .groupby('Group')
            .head(3))

## 6) Window features: rolling / expanding / shift
Sort properly first.

In [ ]:
# Pandas windows cheat-sheet: LAG/LEAD (shift) vs rolling (rows/time) vs expanding
# Copy/paste into your review notebook.

import pandas as pd
import numpy as np

# ---- demo data ----
df = pd.DataFrame({
    "user_id": [1,1,1,1, 2,2,2],
    "ts": [
        "2026-01-01","2026-01-02","2026-01-10","2026-01-21",
        "2026-01-03","2026-01-05","2026-01-20"
    ],
    "revenue": [10, 20, 5, 30, 7, 2, 9]
})

# Always convert + sort before window logic
df["ts"] = pd.to_datetime(df["ts"])
df = df.sort_values(["user_id", "ts"]).reset_index(drop=True)

# ============================================================
# 1) LEAD/LAG (SQL-style) using shift
# ============================================================
# LAG(1): previous row within each user
df["rev_lag1"] = df.groupby("user_id")["revenue"].shift(1)

# LEAD(1): next row within each user
df["rev_lead1"] = df.groupby("user_id")["revenue"].shift(-1)

# delta vs previous
df["rev_delta"] = df["revenue"] - df["rev_lag1"]

# pct change vs previous (careful with NaN / zero)
df["rev_pct_change"] = df["revenue"] / df["rev_lag1"] - 1

# ============================================================
# 2) Rolling window over last k ROWS (row-based)
# ============================================================
# IMPORTANT: rolling(7) uses an integer => window = last 7 rows in that group
df["rev_roll3_rows_sum"] = (
    df.groupby("user_id")["revenue"]
      .rolling(window=3, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df["rev_roll3_rows_mean"] = (
    df.groupby("user_id")["revenue"]
      .rolling(window=3, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)


#better implementation.
df["rev_roll3_rows_mean"] = (
    df.groupby("user_id")["revenue"]
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)


# ============================================================
# 3) Rolling window over last X TIME (time-based)
# ============================================================
# KEY IDEA:
# rolling("7D") uses a STRING OFFSET => window = last 7 days by timestamp,
# NOT last 7 rows. This requires a DatetimeIndex.
tmp = df.set_index("ts")

df["rev_roll7d_sum"] = (
    tmp.groupby("user_id")["revenue"]
       .rolling(window = "7D", min_periods=1)        # time-duration window
       .sum()
       .reset_index(level=0, drop=True)
       .values                               # align back to df rows
)

df["rev_roll7d_mean"] = (
    tmp.groupby("user_id")["revenue"]
       .rolling("7D", min_periods=1)
       .mean()
       .reset_index(level=0, drop=True)
       .values
)

# ============================================================
# 4) Expanding window (cumulative from start to current row)
# ============================================================
df["rev_cumsum"] = df.groupby("user_id")["revenue"].cumsum()

df["rev_expanding_mean"] = (
    df.groupby("user_id")["revenue"]
      .expanding(min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

# ============================================================
# 5) Why rolling("7D") is time-based, not row-based (tiny intuition)
# ============================================================
# rolling(3) => includes last 3 records, even if they are far apart in time
# rolling("7D") => includes only records whose timestamps fall in (t-7days, t]
# For sparse events, time-based window can include fewer (or zero) rows.

print(df)


## 8) String operations (`.str`)

In [ ]:
s = df["text"].astype("string")
 
# contains
mask = s.str.contains("refund", case=False, na=False)
df_refund = df.loc[mask]

# extract group
df["order_id"] = s.str.extract(r"order[:\s]+(\d+)", expand=False)

# split
df[["first","last"]] = s.str.split(" ", n=1, expand=True)

# replace non-digits
df["digits"] = df["phone"].astype("string").str.replace(r"\D+", "", regex=True)

# len
df["len"] = s.str.len()

## 9) Reshaping: pivot/pivot_table/melt + flatten MultiIndex columns

In [2]:
import pandas as pd
demo = pd.DataFrame({
    "user_id": [1,1,1,2,2],
    "event":   ["play","play","like","play","like"],
    "revenue": [10, 15, 1, 7, 2],
    "dur":     [30, 40, 0, 25, 0],
})

# pivot_table often creates MultiIndex columns when:
# - multiple values (revenue, dur)
# - multiple aggfuncs (sum, mean)
# - pivot_table output is unique on rows by index and duplicates are handled via aggfunc
pt = demo.pivot_table(index="user_id",
                      columns="event",
                      values=["revenue","dur"],
                      aggfunc=["sum","mean"],
                      fill_value=0)
pt

sum                   mean                    
         dur      revenue       dur       revenue      
event   like play    like play like  play    like  play
user_id                                                
1          0   70       1   25  0.0  35.0     1.0  12.5
2          0   25       2    7  0.0  25.0     2.0   7.0

In [3]:
### Flatten columns to single level

pt.columns = ['_'.join( col) for col in pt.columns]
# pt.columns = ['_'.join(map(str, col)).strip() for col in pt.columns]
pt.head()

,sum_dur_like,sum_dur_play,sum_revenue_like,sum_revenue_play,mean_dur_like,mean_dur_play,mean_revenue_like,mean_revenue_play
user_id,,,,,,,,
1,0,70,1,25,0.0,35.0,1.0,12.5
2,0,25,2,7,0.0,25.0,2.0,7.0


### Melt (wide -> long) pattern

In [ ]:
# wide -> long
import pandas as pd

df = pd.DataFrame({
    "Product": ["A", "B"],
    "Jan_2026": [100, 150],
    "Feb_2026": [120, 160]
})

df_long = pd.melt(
    df, 
    id_vars=["Product"], 
    var_name="Month", 
    value_name="Sales"
)
#output
# Product	Month	Sales
# 0	A	Jan_2026	100
# 1	B	Jan_2026	150
# 2	A	Feb_2026	120
# 3	B	Feb_2026	160

## 10) Joins / concat

In [ ]:
# Merge / join
df = df.merge(dim_users, on="user_id", how="left")

# Validate to avoid join explosions when you expect many-to-one
df = df.merge(dim_users, on="user_id", how="left", validate="many_to_one")

# Concat
df = pd.concat([df1, df2], axis=0, ignore_index=True)

#horizontal
result = pd.concat([df1.reset_index(drop=True), df2.reset_index(drop=True)], axis=1)

## 11) Plotting (quick)

In [ ]:
# Histogram
df["revenue"].plot(kind="hist", bins=30); plt.show()

# Line: Scatter
df.plot(x="x", y="y", kind="line") #kind = 'scatter'
plt.show()
# Bar: top categories
df["country"].value_counts().head(10).plot(kind="bar"); plt.show()

# Scatter
df.plot(x="feature1", y="revenue",kind="scatter",  alpha=0.5); plt.show()

## 7) Dates: increment/decrement + month boundaries

In [4]:
# Event-stream Pandas: top .dt patterns (with concise clarifications + tiny running example)
import pandas as pd

# --- tiny example (remove in interview if df already exists) ---
df = pd.DataFrame({
    "user_id": [1,1,1,2,2],
    "ts": ["2026-01-01 10:15", "2026-01-01 10:55", "2026-01-02 00:05",
           "2026-01-01 23:59", "bad_ts"],
    "revenue": [10, 20, 5, 7, 2]
})

# 0) Parse timestamps (bad rows -> NaT), then sort (critical for shift/rolling)
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
df = df.sort_values(["user_id", "ts"])

# 1) Date key (daily DAU / retention). .dt.date => python date objects
df["date"] = df["ts"].dt.date

# 2)keeps date+hour Hour BUCKET (timestamp bin) for daily/hourly/minute aggregation; 
#    floor("H") makes 10:15 -> 10:00, 10:55 -> 10:00, 00:05 -> 00:00
df["hour_bucket"] = df["ts"].dt.floor("H")

# 3) Hour of day FEATURE (0..23). Different from hour_bucket (loses the date)
df["hour_of_day"] = df["ts"].dt.hour

# 4) Day-of-week feature (Mon=0..Sun=6) + weekend flag
df["dow"] = df["ts"].dt.dayofweek
df["is_weekend"] = df["dow"].isin([5, 6])

# 5) Week/month buckets (calendar-aware). Prefer to_period for W/M over floor.
#    Use explicit week definition if you care (e.g., W-MON means weeks ending Monday).
df["week_bucket"] = df["ts"].dt.to_period("W").dt.start_time
df["month_bucket"] = df["ts"].dt.to_period("M").dt.start_time

# 6) LAG/LEAD style: previous event timestamp per user + gap in seconds
df["prev_ts"] = df.groupby("user_id")["ts"].shift(1)
df["gap_sec"] = (df["ts"] - df["prev_ts"]).dt.total_seconds()

# 7) First event of day per user (streak/session logic): (normalize Sets timestamp to 00:00:00)
prev_day = df.groupby("user_id")["ts"].shift(1).dt.normalize()
df["is_first_event_of_day"] = df["ts"].dt.normalize().ne(prev_day)

# 8) Range filter (e.g., last 7 days) — ignores NaT automatically in comparisons
cutoff = pd.Timestamp.now() - pd.Timedelta(days=7)
df_last7d = df[df["ts"] >= cutoff]

# 9) (Optional) timezone: if ts is tz-naive but represents UTC, localize then convert
# df["ts_utc"] = df["ts"].dt.tz_localize("UTC")
# df["ts_local"] = df["ts_utc"].dt.tz_convert("Asia/Kolkata")

print(df[["user_id","ts","date","hour_bucket","hour_of_day","dow","week_bucket","month_bucket","prev_ts","gap_sec","is_first_event_of_day","is_weekend"]])


   user_id                  ts        date         hour_bucket  hour_of_day  \
0        1 2026-01-01 10:15:00  2026-01-01 2026-01-01 10:00:00         10.0   
1        1 2026-01-01 10:55:00  2026-01-01 2026-01-01 10:00:00         10.0   
2        1 2026-01-02 00:05:00  2026-01-02 2026-01-02 00:00:00          0.0   
3        2 2026-01-01 23:59:00  2026-01-01 2026-01-01 23:00:00         23.0   
4        2                 NaT         NaT                 NaT          NaN   

   dow week_bucket month_bucket             prev_ts  gap_sec  \
0  3.0  2025-12-29   2026-01-01                 NaT      NaN   
1  3.0  2025-12-29   2026-01-01 2026-01-01 10:15:00   2400.0   
2  4.0  2025-12-29   2026-01-01 2026-01-01 10:55:00  47400.0   
3  3.0  2025-12-29   2026-01-01                 NaT      NaN   
4  NaN         NaT          NaT 2026-01-01 23:59:00      NaN   

   is_first_event_of_day  is_weekend  
0                   True       False  
1                  False       False  
2                   Tru

## 12) Gotchas checklist
- Chained assignment → `.loc[...]` + `.copy()`
- `COUNT(col)` ignores nulls; `value_counts(dropna=False)` when NaNs matter
- `merge` can multiply rows if keys aren’t unique → use `validate=...`
- Window ops require correct sort + understanding row-window vs time-window
- Prefer `agg/transform` over `apply` unless necessary

# 13) Mini Drills (warm-up)
Try from memory, then use the solution patterns.

### Drill 1 — Top-2 rows by revenue per user

In [ ]:
# top2 = (df.sort_values(["user_id","revenue"], ascending=[True, False])
#           .groupby("user_id").head(2))

### Drill 2 — Rolling 7 row mean revenue per user

In [ ]:
# df = df.sort_values(["user_id","ts"])
# df["roll7_mean_rev"] = (df.groupby("user_id")["revenue"]
#                           .rolling(7, min_periods=1).mean()
#                           .reset_index(level=0, drop=True))

### Drill 3 — Pivot + flatten columns

In [ ]:
# pt = df.pivot_table(index="user_id", columns="event", values="revenue", aggfunc=["sum","mean"], fill_value=0)
# pt.columns = ['_'.join(map(str, col)).strip() for col in pt.columns]

### Drill 4 — Extract order_id from text

In [ ]:
# df["order_id"] = df["text"].astype("string").str.extract(r"order[:\s]+(\d+)", expand=False)